In [215]:
### Create a sample text file
import os

os.makedirs("../data/text_files", exist_ok=True)


In [216]:
sample_texts = {
    "../data/text_files/sample1.txt": (
        "Acme AI Solutions is a fictional company that provides machine learning consulting, chatbot development, and document intelligence services for small businesses.\n\n"
        "The company was founded in 2021 and is headquartered in London. Its main product is called InsightBot, an assistant that helps users search internal documents, summarize reports, and answer questions from company knowledge bases.\n\n"
        "Acme AI Solutions offers three service plans. The Starter plan costs $29 per month and supports up to 1,000 document queries. The Professional plan costs $99 per month and includes analytics, team access, and up to 10,000 queries. The Enterprise plan uses custom pricing and includes priority support, private deployment, and advanced security controls.\n\n"
        "Customer support is available from Monday to Friday, 9:00 AM to 6:00 PM. Enterprise customers also receive weekend emergency support.\n\n"
        "The company uses Python, LangChain, ChromaDB, and sentence-transformer models in many of its internal prototypes. A common demo use case is retrieving answers from PDF files and text documents.\n\n"
        "The refund policy states that monthly subscriptions can be canceled at any time, but annual subscriptions are refundable only within the first 14 days of purchase."
    )
}

for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created.")


Sample text files created.


In [217]:
### TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/sample1.txt", encoding="utf-8")
document = loader.load()
print(document)


[Document(metadata={'source': '../data/text_files/sample1.txt'}, page_content='Acme AI Solutions is a fictional company that provides machine learning consulting, chatbot development, and document intelligence services for small businesses.\n\nThe company was founded in 2021 and is headquartered in London. Its main product is called InsightBot, an assistant that helps users search internal documents, summarize reports, and answer questions from company knowledge bases.\n\nAcme AI Solutions offers three service plans. The Starter plan costs $29 per month and supports up to 1,000 document queries. The Professional plan costs $99 per month and includes analytics, team access, and up to 10,000 queries. The Enterprise plan uses custom pricing and includes priority support, private deployment, and advanced security controls.\n\nCustomer support is available from Monday to Friday, 9:00 AM to 6:00 PM. Enterprise customers also receive weekend emergency support.\n\nThe company uses Python, Lang

In [218]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

# Load all text files from the directory
dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
documents = dir_loader.load()
print(documents)


[Document(metadata={'source': '..\\data\\text_files\\sample1.txt'}, page_content='Acme AI Solutions is a fictional company that provides machine learning consulting, chatbot development, and document intelligence services for small businesses.\n\nThe company was founded in 2021 and is headquartered in London. Its main product is called InsightBot, an assistant that helps users search internal documents, summarize reports, and answer questions from company knowledge bases.\n\nAcme AI Solutions offers three service plans. The Starter plan costs $29 per month and supports up to 1,000 document queries. The Professional plan costs $99 per month and includes analytics, team access, and up to 10,000 queries. The Enterprise plan uses custom pricing and includes priority support, private deployment, and advanced security controls.\n\nCustomer support is available from Monday to Friday, 9:00 AM to 6:00 PM. Enterprise customers also receive weekend emergency support.\n\nThe company uses Python, L

In [219]:
type(documents[0])


langchain_core.documents.base.Document

In [220]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_documents(documents, chunk_size=500, chunk_overlap=50):
    """Split documents into smaller chunks for better RAG performance."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    return split_docs



In [221]:
chunks = split_documents(documents)
chunks


Split 1 documents into 3 chunks


[Document(metadata={'source': '..\\data\\text_files\\sample1.txt'}, page_content='Acme AI Solutions is a fictional company that provides machine learning consulting, chatbot development, and document intelligence services for small businesses.\n\nThe company was founded in 2021 and is headquartered in London. Its main product is called InsightBot, an assistant that helps users search internal documents, summarize reports, and answer questions from company knowledge bases.'),
 Document(metadata={'source': '..\\data\\text_files\\sample1.txt'}, page_content='Acme AI Solutions offers three service plans. The Starter plan costs $29 per month and supports up to 1,000 document queries. The Professional plan costs $99 per month and includes analytics, team access, and up to 10,000 queries. The Enterprise plan uses custom pricing and includes priority support, private deployment, and advanced security controls.\n\nCustomer support is available from Monday to Friday, 9:00 AM to 6:00 PM. Enterpri

In [222]:
### Embeddings and Vector Store
from typing import Any, Dict, List
import uuid

import chromadb
import numpy as np
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer


class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                "Model loaded successfully. Embedding dimension:",
                self.model.get_embedding_dimension(),
            )
        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError("Model is not loaded.")
        return self.model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

    def get_embedding_dimension(self) -> int:
        if self.model is None:
            raise ValueError("Model is not loaded.")
        return self.model.get_embedding_dimension()


embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3531.92it/s]


Model loaded successfully. Embedding dimension: 384


In [223]:
class VectorStore:
    def __init__(
        self,
        collection_name: str = "text_documents_demo",
        persist_directory: str = "../data/vectorstore",
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory

        try:
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"hnsw:space": "cosine"},
            )
            print(f"ChromaDB initialized with collection: {self.collection_name}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(self, documents: List[Document], embeddings: np.ndarray) -> None:
        if not documents:
            print("No documents to add.")
            return

        if embeddings.ndim != 2:
            raise ValueError(f"Embeddings must be 2D. Got shape: {embeddings.shape}")

        if len(documents) != embeddings.shape[0]:
            raise ValueError(
                f"Number of documents ({len(documents)}) and embeddings rows ({embeddings.shape[0]}) must match."
            )

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, doc in enumerate(documents):
            ids.append(str(uuid.uuid4()))

            metadata = dict(doc.metadata) if doc.metadata else {}
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embeddings[i].tolist())

        self.collection.upsert(
            ids=ids,
            documents=documents_text,
            metadatas=metadatas,
            embeddings=embeddings_list,
        )
        print(f"Added {len(documents)} documents to collection '{self.collection_name}'")



In [224]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)

vectorstore = VectorStore()
vectorstore.add_documents(chunks, embeddings)


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.12it/s]


ChromaDB initialized with collection: text_documents_demo
Added 3 documents to collection 'text_documents_demo'


In [225]:
### Retrieval Pipeline
class RAGRetriever:
    def __init__(self, vectorstore: VectorStore, embedding_manager: EmbeddingManager):
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.3) -> List[Dict[str, Any]]:
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        results = self.vectorstore.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,
        )

        retrieved_docs = []
        documents = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]
        distances = results.get("distances", [[]])[0]
        ids = results.get("ids", [[]])[0]

        for doc_id, document, metadata, distance in zip(ids, documents, metadatas, distances):
            similarity_score = 1 - distance
            if similarity_score >= score_threshold:
                retrieved_docs.append(
                    {
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                    }
                )

        return retrieved_docs



In [226]:
retriever = RAGRetriever(vectorstore, embedding_manager)
query = "What is the main product of Acme AI Solutions?"
results = retriever.retrieve(query, top_k=3)

print(f"Query: {query}\n")
if not results:
    print("No relevant results found.")
else:
    for index, result in enumerate(results, start=1):
        print(f"Result {index}")
        print(f"Similarity: {result['similarity_score']:.4f}")
        print(f"Content: {result['content']}")
        print(f"Metadata: {result['metadata']}")
        print("-" * 80)


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.39it/s]

Query: What is the main product of Acme AI Solutions?

Result 1
Similarity: 0.6861
Content: Acme AI Solutions is a fictional company that provides machine learning consulting, chatbot development, and document intelligence services for small businesses.

The company was founded in 2021 and is headquartered in London. Its main product is called InsightBot, an assistant that helps users search internal documents, summarize reports, and answer questions from company knowledge bases.
Metadata: {'doc_index': 0, 'source': '..\\data\\text_files\\sample1.txt', 'content_length': 392}
--------------------------------------------------------------------------------
Result 2
Similarity: 0.4742
Content: Acme AI Solutions offers three service plans. The Starter plan costs $29 per month and supports up to 1,000 document queries. The Professional plan costs $99 per month and includes analytics, team access, and up to 10,000 queries. The Enterprise plan uses custom pricing and includes priority support